# docextract probe on a Colab GPU

Runs one probe configuration from `manifests/probes/` with the same requests and engine as the Modal
probes (`runtimes/modal/probe.py`). PDFs and schemas come from the pinned ExtractBench dataset on
Hugging Face and the model from its pinned, hash-verified revision; nothing private is needed.

1. **Runtime → Change runtime type → A100 GPU** (or L4), then **Runtime → Run all**.
2. Answers are saved to Google Drive (`MyDrive/docextract/<probe>/`) as each one finishes. If the
   session disconnects, run all again: answered requests are skipped.
3. When the run ends (or fails), the last cell saves to Drive and releases the GPU, so it stops
   using compute units. Stop it early with the stop button: the GPU is released the same way.
4. Score locally: copy `answers.json` to `artifacts/probes/<probe>/answers.json` and run
   `uv run python tools/score_probe.py manifests/probes/<probe>.json`.

In [ ]:
PROBE = "c1"  # a file in manifests/probes/
REF = "main"  # the docextract commit or branch to run
MINUTES = None  # stop after this long; None uses the probe's limit

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/docextract/{PROBE}"

In [ ]:
%%bash -s "$REF"
set -e
rm -rf /content/docextract
git clone -q https://github.com/RohithReddy20/docextract /content/docextract
cd /content/docextract
git checkout -q "$1"
git log --oneline -1
pip install -q uv
uv pip install --system -q -r runtimes/colab/requirements.txt

In [ ]:
import json
import subprocess

from google.colab import drive, runtime

name, memory = (
    subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True
    )
    .strip()
    .split(", ")
)
# An L4 (24 GB) runs about 24 answers at once before the KV cache overflows; larger GPUs run more.
engine = {"max_num_seqs": 64} if int(memory) >= 40000 else {}
print(name, memory, "MiB", engine)
args = ["--config", f"manifests/probes/{PROBE}.json", "--out", OUT, "--engine", json.dumps(engine)]
if MINUTES:
    args += ["--minutes", str(MINUTES)]
# The runner's log is shown as it runs.
process = subprocess.Popen(
    ["python", "-u", "runtimes/colab/probe.py", *args],
    cwd="/content/docextract",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
try:
    for line in process.stdout:
        print(line, end="")
    print("exit code", process.wait())
except KeyboardInterrupt:
    # Stopped from the notebook: the runner writes answers.json from what finished, then exits.
    process.terminate()
    print("exit code", process.wait(timeout=300))
finally:
    # Write everything to Drive, then release the GPU so it stops using compute units.
    drive.flush_and_unmount()
    runtime.unassign()